# Notebook 3 - Legal Amount Recognition (PyTorch CNN-BiLSTM-CTC)

## Quick Note
- Purpose: prepare extracted legal amount images and labels for Arabic text recognition.
- The first cells only add a clean PyTorch setup, helpful libraries, and the main working paths.

In [ ]:
%pip install --upgrade pip
%pip install torch torchvision opencv-python pillow numpy pandas matplotlib seaborn scikit-learn tqdm

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.optim as optim
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

paths = {
    "legal_images_dir": repo_root / "data" / "extracted" / "legal",
    "legal_labels_file": repo_root / "data" / "labels" / "legal_labels.csv",
    "tokenization_dir": repo_root / "data" / "tokenization",
    "models_dir": repo_root / "models" / "legal",
    "logs_dir": repo_root / "outputs" / "legal_logs",
}

for d in [paths["tokenization_dir"], paths["models_dir"], paths["logs_dir"]]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for k, v in paths.items():
    print(f"{k}: {v} | exists={v.exists()}")

print("Legal setup is ready.")


In [ ]:
preprocessing_config = {
    "resize_width": 1340,
    "resize_height": 140,
    "normalize_min": -1,
    "normalize_max": 1,
    "horizontal_flip_for_rtl_alignment": True,
    "max_label_sequence_length": 84,
    "max_original_length_reported": 71,
}

tokenization_config = {
    "strategies": [
        "subword",
        "enhanced_subword",
        "word",
        "enhanced_word",
        "character",
    ],
    "character_class_count": 24,
}

training_config = {
    "epochs": 500,
    "early_stopping_patience": 20,
    "learning_rate_start": 1e-4,
    "lr_decay_factor": 0.9,
    "lr_decay_every_steps": 10000,
    "split_train": 0.85,
    "split_val": 0.15,
}

print(preprocessing_config)
print(tokenization_config)
print(training_config)


In [ ]:
# Optional: add data augmentation pipeline setup (zoom-in, left-rotation, right-rotation, elastic transform).
print("Legal phase setup complete.")